# D — SSVI Implied-Volatility Surface Risk Framework

**SSVI Volatility Surface Dynamics — S&P 500 Options (2010–2020)**
Politecnico di Milano — Econometrics Project (A.Y. 2025/26)

---

## Overview

This notebook constructs a **volatility-surface risk framework** and applies it to a
risk-adjusted quoting illustration using daily SPX options data (notebooks A/B/C).

This notebook is **not** a full market-making model.
No inventory dynamics, order-flow modelling, or observed bid-ask calibration are included.

**Research question:**
> *When does forecasting the realized volatility of the SSVI implied-volatility surface
> provide incremental residual risk coverage beyond a vol-scaled baseline quoting rule?*

The objective is to study whether IV-surface RV forecasting provides material incremental
coverage beyond a parsimonious baseline, and under what conditions that contribution holds.

**Two distinct volatility concepts:**

| Concept | Source | Definition |
|---------|--------|-----------|
| Price-based RV | Notebook C | RV of log-returns of the underlying |
| **IV-surface RV** (`surface_move`) | **This notebook** | RMS of daily ΔIV across the SSVI grid |

`surface_move` measures the *realized volatility of the implied-volatility surface*
(volatility-of-volatility) — not the realized volatility of the underlying.

**Pipeline:**
```
data/ssvi_all_dates_clean_results.csv          ← SSVI calibration (notebook B)
  → IV surface reconstruction (45-point grid)
  → ΔIV daily first differences → surface_move (IV-surface RV)
  → HAR-J forecast of surface_move (train-only OLS)
  → c*(ES₉₅) residual-risk calibration on validation set
  → spread_final = spread_AS + c* × RV̂
  → coverage evaluation and baseline-sensitivity analysis on test set
```

**Baseline spread — parsimonious vol-scaled proxy:**

$$\text{spread}_{AS}(k,T,t) = \gamma \times \sigma_{ATM}(T,t) \times \sqrt{\Delta t} \times (1 + \kappa|k|)$$

Inspired by Avellaneda-Stoikov (2008) structure but **not** an implementation of that model.
`γ` and `κ` are **fixed heuristic parameters** — not calibrated to observed spreads, inventory, or order flow.

**Sections:**

| # | Title |
|---|-------|
| 1 | Setup |
| 2 | IV Surface Reconstruction |
| 3 | Surface RV Definition and Properties |
| 4 | HAR-J Forecasting |
| 5 | Residual-Risk Addon Calibration |
| 6 | Coverage Evaluation |
| **7** | **Baseline Sensitivity (γ) — main result** |
| 8 | Summary and Conclusions |
| A.1 | Appendix: PCA on ΔIV (supplementary) |
| A.2 | Appendix: HMM Regime Detection (supplementary) |
| A.3 | Appendix: Vega-Weighted Robustness (supplementary) |


## 1. Setup

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as smapi

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ── Path resolution (works from econometric_analysis/, notebook/, or repo root) ──
BASE = Path.cwd().resolve()
if BASE.name in ('econometric_analysis', 'notebook'):
    BASE = BASE.parent

SRC_DIR    = BASE / 'src'
DATA_DIR   = BASE / 'data'
OUTPUT_DIR = BASE / 'output'
PLOT_DIR   = OUTPUT_DIR / 'plots'
CACHE_DIR  = OUTPUT_DIR / 'cache'

for _d in (OUTPUT_DIR, PLOT_DIR, CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ssvi_mm_risk_engine import (
    get_grid_metadata, load_ssvi_results, build_iv_panel,
    compute_delta_iv_panel, compute_surface_move, compute_move_by_maturity,
    build_future_rv_targets, chronological_split, ensure_dirs,
    make_har_features, detect_jumps_bpv, choose_jump_method,
    fit_har_model, qlike_loss, diebold_mariano_test, compute_forecast_metrics,
    fit_train_pca, transform_pca_splits, save_pca_outputs,
    fit_hmm_regimes, summarize_hmm_regimes,
    compute_avellaneda_style_spread, expected_shortfall, calibrate_c_es95,
    compute_spread_final, backtest_spread_coverage, compute_vega_weighted_robustness,
    plot_surface_move, plot_move_by_maturity, plot_jump_detection_comparison,
    plot_pca_loadings, plot_pca_scores, plot_hmm_regimes, plot_forecast_global,
    plot_cstar_term_structure, plot_spread_decomposition,
)

NB = 'D'   # output-file prefix for all saved files

# ── Grid ──────────────────────────────────────────────────────────────────────
K_GRID = np.array([-0.30, -0.20, -0.10, -0.05, 0.00, 0.05, 0.10, 0.20, 0.30])
T_GRID = np.array([1/12, 3/12, 6/12, 1.0, 2.0])
k_flat, t_flat, GRID_COLS, T_LABELS, MAT_T_MAP = get_grid_metadata(K_GRID, T_GRID)

BUCKET_COLS = {
    'left_wing':  [c for c, kv in zip(GRID_COLS, k_flat) if kv <= -0.10],
    'atm':        [c for c, kv in zip(GRID_COLS, k_flat) if abs(kv) <= 0.05],
    'right_wing': [c for c, kv in zip(GRID_COLS, k_flat) if kv >= 0.10],
}

# ── Strategy parameters ───────────────────────────────────────────────────────
GAMMA_AS    = 0.50   # risk-aversion proxy (heuristic; uncalibrated)
KAPPA_K     = 0.50   # wing-widening factor
DT          = 1 / 252
CALIB_ALPHA = 0.95
HORIZONS    = [5, 10, 22]
H_MAIN      = 5

print(f'Backend   : {matplotlib.get_backend()}')
print(f'BASE      : {BASE}')
print(f'DATA_DIR  : {DATA_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'CACHE_DIR : {CACHE_DIR}')
print(f'Grid      : {len(K_GRID)} k × {len(T_GRID)} T = {len(GRID_COLS)} points')


## 2. IV Surface Reconstruction

SSVI calibration parameters are loaded from `data/ssvi_all_dates_clean_results.csv`.
Dates with fewer than 100 observations after cleaning are excluded at calibration time;
this cell documents the exclusion pattern before building the surface.

The IV panel (45-point grid, 9 strikes × 5 maturities) is loaded from
`output/cache/ssvi_surface_iv_panel.csv` if cached, otherwise reconstructed from
SSVI parameters on first run and saved to cache.

Safety checks confirm DatetimeIndex, monotone ordering, and index alignment.

In [ ]:
import re as _re

_raw = pd.read_csv(DATA_DIR / 'ssvi_all_dates_clean_results.csv')
_failed = _raw[_raw['n_obs'] == 0].copy()
_failed['actual_n'] = _failed['message'].apply(
    lambda m: int(x.group(1)) if (x := _re.search(r'Only (\d+) obs', str(m))) else None
)

print('=== CALIBRATION EXCLUSION DIAGNOSTIC ===')
print(f'Total dates in CSV      : {len(_raw)}')
print(f'Successful calibrations : {(_raw["n_obs"] > 0).sum()}')
print(f'Failed (n_obs = 0)      : {len(_failed)}  ({len(_failed)/len(_raw)*100:.1f}%)')
print()
print('Failed dates by time_elapsed range:')
_bins  = [0, 50, 100, 200, 500, 5000]
_lbls  = ['0-50', '50-100', '100-200', '200-500', '500+']
_failed['te_bucket'] = pd.cut(_failed['time_elapsed'], bins=_bins, labels=_lbls)
print(_failed['te_bucket'].value_counts().sort_index().to_string())
print()
print('Actual obs in failed rows:')
print(f'  min={_failed["actual_n"].min()}  median={_failed["actual_n"].median():.0f}  max={_failed["actual_n"].max()}')
print()
print('Threshold sensitivity (dates recovered if threshold were lowered):')
for _t in [40, 60, 80]:
    _r = (_failed['actual_n'] >= _t).sum()
    print(f'  threshold = {_t}: +{_r} dates ({_r/len(_raw)*100:.1f}% of dataset)')
print()
print('Recommendation: keep threshold at 100.')
print('  First ~103 trading days (Jan–Apr 2010) are systematically sparse (~58 obs/day).')
print('  47 borderline dates (80–97 obs) are scattered anomalies; recovering them')
print('  does not materially change the 2,653-day panel.')


In [ ]:
# ── Load calibration parameters ───────────────────────────────────────────────
params_ok = load_ssvi_results(DATA_DIR, CACHE_DIR)

assert isinstance(params_ok.index, pd.DatetimeIndex), \
    'params_ok.index must be DatetimeIndex'
assert params_ok.index.is_monotonic_increasing, \
    'params_ok.index is not sorted'
assert not params_ok.index.duplicated().any(), \
    f'{params_ok.index.duplicated().sum()} duplicate dates in params_ok'

print(f'Successful calibrations : {len(params_ok)}')
print(f'Date range              : {params_ok.index[0].date()} → {params_ok.index[-1].date()}')
_p = [c for c in ('alpha', 'beta', 'rho', 'eta', 'gamma') if c in params_ok.columns]
print(params_ok[_p].describe().round(4).to_string())

# ── Reconstruct IV panel ───────────────────────────────────────────────────────
_iv_cache = CACHE_DIR / 'ssvi_surface_iv_panel.csv'
iv_panel = build_iv_panel(
    params_ok, k_flat, t_flat, GRID_COLS,
    src_path = _iv_cache,
    out_path = _iv_cache if not _iv_cache.exists() else CACHE_DIR / f'{NB}_iv_panel.csv',
)

assert isinstance(iv_panel.index, pd.DatetimeIndex), 'iv_panel.index must be DatetimeIndex'
assert iv_panel.index.is_monotonic_increasing, 'iv_panel.index is not sorted'
assert iv_panel.index.equals(params_ok.index), \
    f'iv_panel index does not match params_ok ({len(iv_panel)} vs {len(params_ok)} rows)'

print(f'\nIV panel : {iv_panel.shape}  NaN={iv_panel.isnull().mean().mean()*100:.2f}%')
print(f'Dates    : {iv_panel.index[0].date()} → {iv_panel.index[-1].date()}')
print(f'ATM 3M mean IV: {iv_panel["iv_k_0.00_T_0.25"].mean():.4f}')


## 3. Surface RV Definition and Properties

**surface_move(t)** = equal-weighted RMS of daily ΔIV across all 45 grid points —
the *realized volatility of the IV surface* (volatility-of-volatility).
This is the core quantity of the pipeline and must not be confused with price-based RV.

**Jump detection:** BPV (Barndorff-Nielsen & Shephard) produces an implausibly high
jump rate (~31%) on IV-surface moves. A rolling-threshold method is used instead,
targeting a 5–15% jump rate more consistent with the literature.
Method A = rolling q₉₅ + 2σ; Method B = rolling q₉₉. Auto-selector picks whichever
falls in the target range, closest to 10%.

**Splits:** 70% train / 15% val / 15% test (strictly chronological, no shuffle).

### `surface_move` vs price-based RV — a critical distinction

`surface_move(t)` measures the **realized volatility of the implied-volatility surface**
on day *t*: equal-weighted RMS of daily IV changes across all 45 grid points.

$$\text{surface\_move}(t) = \sqrt{\frac{1}{45}\sum_{k,T} \Delta\text{IV}(k,T,t)^2}$$

This is *volatility-of-volatility* — how much the IV surface itself moves per day —
and must not be confused with the price-based RV of notebook C.

| Quantity | Units | Captures |
|----------|-------|---------|
| Price-based RV (notebook C) | vol-units | Return dispersion of the underlying |
| `surface_move` (this notebook) | vol-units | Dispersion of the IV surface itself |

A parallel shift in the IV surface with no underlying movement contributes to
`surface_move` but not to price-based RV.

In [ ]:
delta_iv    = compute_delta_iv_panel(iv_panel)
smove       = compute_surface_move(delta_iv)
move_by_mat = compute_move_by_maturity(delta_iv, GRID_COLS, T_LABELS, MAT_T_MAP)
targets_df  = build_future_rv_targets(smove, move_by_mat, HORIZONS, T_LABELS)

assert delta_iv.index.equals(iv_panel.index[1:]), \
    'delta_iv.index does not equal iv_panel.index[1:] — unexpected NaN rows in IV panel'

pd.concat([smove, move_by_mat], axis=1).to_csv(CACHE_DIR / f'{NB}_surface_moves.csv')
print(f'delta_iv    : {delta_iv.shape}')
print(f'surface_move: mean={smove.mean():.5f}  std={smove.std():.5f}')
print(f'targets     : {targets_df.shape}  non-NaN 5d: {targets_df["future_rv_5d"].notna().sum()}')

plot_surface_move(smove, PLOT_DIR, prefix=NB)
plot_move_by_maturity(move_by_mat, PLOT_DIR, prefix=NB)

split_train, split_val, split_test = chronological_split(smove.index, 0.70, 0.15)
print(f'Train : {len(split_train)}  [{split_train[0].date()} → {split_train[-1].date()}]')
print(f'Val   : {len(split_val)}   [{split_val[0].date()}  → {split_val[-1].date()}]')
print(f'Test  : {len(split_test)}   [{split_test[0].date()}  → {split_test[-1].date()}]')

assert split_train[-1] < split_val[0],  'Train/val overlap detected'
assert split_val[-1]   < split_test[0], 'Val/test overlap detected'


In [ ]:
sm_sq   = smove.pow(2)
j_flag  = detect_jumps_bpv(smove)
feats_g = make_har_features(smove, j_flag=j_flag)

log_rv1  = feats_g['log_rv1'];  log_rv5   = feats_g['log_rv5']
log_rv22 = feats_g['log_rv22']; log_ewma  = feats_g['log_ewma']
j_cnt22  = feats_g['j_cnt22'];  log_j22   = feats_g['log_j22']
log_c5   = feats_g['log_c5'];   log_j5    = feats_g['log_j5']

mat_feats = {}
for t_lbl in T_LABELS:
    ms = move_by_mat.get(f'move_{t_lbl}')
    if ms is not None:
        mat_feats[t_lbl] = make_har_features(ms, j_flag=detect_jumps_bpv(ms))

old_jump_pct = float(j_flag.mean() * 100)
print(f'Jump rate (BPV): {old_jump_pct:.1f}%  → rolling-threshold will select a stricter method.')


In [ ]:
j_flag_old   = j_flag.copy()
j_flag_new, selected_key, selected_pct, cands = choose_jump_method(smove)
pct_A, pct_B = cands['A'][1], cands['B'][1]
print(f'Jump detection:')
print(f'  BPV (original)    : {old_jump_pct:.1f}%')
print(f'  Method A (q95+2σ) : {pct_A:.1f}%')
print(f'  Method B (q99)    : {pct_B:.1f}%')
print(f'  Selected          : Method {selected_key}  ({selected_pct:.1f}%)')

pd.DataFrame(
    {'bpv_jump': j_flag_old, 'method_A': cands['A'][0],
     'method_B': cands['B'][0], 'selected': j_flag_new},
    index=smove.index,
).to_csv(CACHE_DIR / f'{NB}_jump_comparison.csv')

plot_jump_detection_comparison(smove, j_flag_old, j_flag_new,
                                selected_key, old_jump_pct, selected_pct,
                                PLOT_DIR, prefix=NB)

j_flag  = j_flag_new.copy()
feats_g = make_har_features(smove, j_flag=j_flag)
log_rv1  = feats_g['log_rv1'];  log_rv5   = feats_g['log_rv5']
log_rv22 = feats_g['log_rv22']; log_ewma  = feats_g['log_ewma']
j_cnt22  = feats_g['j_cnt22'];  log_j22   = feats_g['log_j22']
log_c5   = feats_g['log_c5'];   log_j5    = feats_g['log_j5']

_use_q99 = (selected_key == 'B')
for t_lbl in T_LABELS:
    ms = move_by_mat.get(f'move_{t_lbl}')
    if ms is None or t_lbl not in mat_feats:
        continue
    _rq = ms.rolling(252, min_periods=50).quantile(0.99 if _use_q99 else 0.95)
    _rs = ms.rolling(252, min_periods=50).std()
    j_fl_m = ((ms > _rq).astype(float).fillna(0.0) if _use_q99
              else (ms > (_rq + 2 * _rs)).astype(float).fillna(0.0))
    mat_feats[t_lbl].update(make_har_features(ms, j_flag=j_fl_m))

print(f'\nJump rate: {old_jump_pct:.1f}% → {selected_pct:.1f}%')


## 4. HAR-J Forecasting

**Models:** Persistence · HAR-RV · HAR-J · HAR-CJ.
All models are fitted by OLS on the **train set only** and evaluated on the test set.
No leakage: the validation set is not used for model selection or fitting.

**Target:** `log(future_rv_5d)` — 5-day forward realized volatility of `surface_move`.

**HAR-J** retains jump components (log_j22, j_cnt22) for structural decomposition:
even if the DM test does not reject equal MSE, the coefficient on jump features
provides an interpretable measure of the tail-risk contribution to surface RV.

**Evaluation metrics:**
- MAE and RMSE on log scale
- QLIKE: loss function robust to outliers in vol forecasting
- Diebold-Mariano (1995) test vs HAR baseline (Newey-West HAC variance, h=5)

**Walk-forward HAR-J:** re-estimated every 22 days on all available data up to the
prediction date (minimum 250 observations). Documents out-of-sample stability.

In [ ]:
feat_global = pd.DataFrame({
    'log_rv1': log_rv1, 'log_rv5': log_rv5, 'log_rv22': log_rv22,
    'log_j22': log_j22, 'j_cnt22': j_cnt22,
    'log_c5':  log_c5,  'log_j5':  log_j5,
})

LOG_TGT        = 'log_future_rv_5d'
log_tgt_global = np.log(targets_df['future_rv_5d'] + 1e-8).rename(LOG_TGT)

HAR_COLS   = ['log_rv1', 'log_rv5', 'log_rv22']
HARJ_COLS  = ['log_rv1', 'log_rv5', 'log_rv22', 'log_j22', 'j_cnt22']
HARCJ_COLS = ['log_c5',  'log_j5',  'log_rv22']
PERS_COLS  = ['log_rv22']

TARGETS_TO_FIT = {'global': log_tgt_global}
for t_lbl in T_LABELS:
    col = f'future_rv_{H_MAIN}d_{t_lbl}'
    if col in targets_df.columns:
        TARGETS_TO_FIT[t_lbl] = np.log(targets_df[col] + 1e-8).rename(f'log_tgt_{t_lbl}')

all_metrics, fc_test, fc_val = [], {}, {}

for tgt_key, log_tgt in TARGETS_TO_FIT.items():
    fbase = feat_global if tgt_key == 'global' else pd.DataFrame(mat_feats.get(tgt_key, {}))
    if fbase.empty:
        continue
    df_all = pd.concat([log_tgt, fbase], axis=1)
    for mname, fcols in [('Pers',   PERS_COLS),
                          ('HAR',    HAR_COLS),
                          ('HAR-J',  HARJ_COLS),
                          ('HAR-CJ', HARCJ_COLS)]:
        avail = [c for c in fcols if c in df_all.columns]
        if not avail:
            continue
        fc_te = fit_har_model(df_all, avail, log_tgt.name, split_train, split_test)
        fc_va = fit_har_model(df_all, avail, log_tgt.name, split_train, split_val)
        y_te  = log_tgt.reindex(split_test).dropna()
        common = y_te.index.intersection(fc_te.dropna().index)
        if len(common) < 5:
            continue
        m = compute_forecast_metrics(y_te.loc[common].values, fc_te.loc[common].values)
        m.update({'model': mname, 'target': tgt_key})
        all_metrics.append(m)
        fc_test[f'{mname}|{tgt_key}'] = fc_te
        fc_val[f'{mname}|{tgt_key}']  = fc_va

for tgt_key, log_tgt in TARGETS_TO_FIT.items():
    har_fc  = fc_test.get(f'HAR|{tgt_key}')
    harj_fc = fc_test.get(f'HAR-J|{tgt_key}')
    if har_fc is None or harj_fc is None:
        continue
    y_te   = log_tgt.reindex(split_test).dropna()
    common = (y_te.index
              .intersection(har_fc.dropna().index)
              .intersection(harj_fc.dropna().index))
    if len(common) < 20:
        continue
    dm_s, dm_p = diebold_mariano_test(
        y_te.loc[common] - har_fc.loc[common],
        y_te.loc[common] - harj_fc.loc[common], h=H_MAIN)
    for m in all_metrics:
        if m['target'] == tgt_key and m['model'] == 'HAR-J':
            m['DM_stat_vs_HAR'] = dm_s
            m['DM_pval_vs_HAR'] = dm_p

REFIT_FREQ, MIN_TRAIN = 22, 250
df_wf = pd.concat([log_tgt_global, feat_global[HARJ_COLS]], axis=1).dropna()
fc_wf, _ols_wf, _X_tr = {}, None, None
for i, dt in enumerate(split_test):
    if dt not in df_wf.index:
        continue
    pos = int(np.searchsorted(df_wf.index, dt))
    if pos < MIN_TRAIN:
        continue
    if i % REFIT_FREQ == 0 or _ols_wf is None:
        _X_tr = smapi.add_constant(df_wf.iloc[:pos][HARJ_COLS], has_constant='add')
        _y_tr = df_wf.iloc[:pos][LOG_TGT]
        _ols_wf = smapi.OLS(_y_tr, _X_tr).fit()
    row = smapi.add_constant(df_wf.loc[[dt]][HARJ_COLS], has_constant='add')
    row = row.reindex(columns=_X_tr.columns, fill_value=0.0)
    fc_wf[dt] = float(_ols_wf.predict(row)[0])

fc_wf_ser = pd.Series(fc_wf, name='HAR-J (WF)')
fc_test[f'HAR-J|global|WF'] = fc_wf_ser

metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(OUTPUT_DIR / f'{NB}_forecast_metrics.csv', index=False)
pd.DataFrame({k: v for k, v in fc_test.items()}).to_csv(OUTPUT_DIR / f'{NB}_forecasts.csv')

print('=== FORECAST TABLE (global, h=5, test set) ===')
_cols = ['model', 'n', 'MAE_log', 'RMSE_log', 'QLIKE', 'DM_stat_vs_HAR', 'DM_pval_vs_HAR']
print(metrics_df[metrics_df['target'] == 'global']
      [[c for c in _cols if c in metrics_df.columns]].round(4).to_string(index=False))

plot_forecast_global(targets_df, fc_test, fc_wf_ser, split_test, H_MAIN, PLOT_DIR, prefix=NB)


## 5. Residual-Risk Addon Calibration

### 5.1  Baseline spread

$$\text{spread}_{AS}(k,T,t) = \gamma \times \sigma_{ATM}(T,t) \times \sqrt{\Delta t} \times (1 + \kappa|k|)$$

`GAMMA_AS = 0.5` and `KAPPA_K = 0.5` are **heuristic, uncalibrated** parameters —
not estimated from observed spreads, inventory dynamics, or order flow.
The choice of γ is the central sensitivity parameter; Section 7 examines this dependence.

### 5.2  Residual-risk addon — c* calibration (validation set only)

$$c^*(\text{bucket}) = \text{ES}_{0.95}\!\left(\frac{\max(\text{actual\_rv} - \text{spread}_{AS},\, 0)}{\hat{\text{RV}}}\right)$$

`c*` measures the tail-risk residual of the baseline: how many units of the HAR-J
surface-RV forecast are needed to cover the 95th-percentile shortfall days.

- Large `c*`: the baseline is tight relative to realised surface risk → addon is material.
- Small `c*`: the baseline absorbs most surface risk → addon is economically negligible.

How `c*` responds to γ is the central question of Section 7.

*Calibration uses the validation set only. Test set is held out for coverage evaluation.*

In [ ]:
spread_AS = compute_avellaneda_style_spread(
    iv_panel, GRID_COLS, k_flat, MAT_T_MAP, GAMMA_AS, KAPPA_K, DT
)
print(f'Baseline parameters (heuristic, uncalibrated): GAMMA_AS={GAMMA_AS}  KAPPA_K={KAPPA_K}  DT=1/252')
for bucket, cols in BUCKET_COLS.items():
    mv = float(spread_AS[cols].mean().mean())
    print(f'  {bucket:12s}: {mv:.5f} vol-units')
print(f'  mean surface_move : {smove.mean():.5f} vol-units')


In [ ]:
calib_targets = {'global': ('future_rv_5d', spread_AS.mean(axis=1))}
for t_lbl, t_str in MAT_T_MAP.items():
    tgt_col  = f'future_rv_{H_MAIN}d_{t_lbl}'
    mat_cols = [c for c in GRID_COLS if f'_T_{t_str}' in c]
    if tgt_col in targets_df.columns and mat_cols:
        calib_targets[t_lbl] = (tgt_col, spread_AS[mat_cols].mean(axis=1))

calib_df = calibrate_c_es95(
    split_val, targets_df, calib_targets, fc_val, log_ewma, CALIB_ALPHA
)
calib_df.to_csv(OUTPUT_DIR / f'{NB}_c_es95_calibration.csv')
print('c* = residual-risk addon coefficient (val set, ES₉₅ of shortfall ratio):')
print(calib_df.round(3).to_string())


### 5.3  c* Term Structure

OLS regression of c*(T) on √T checks whether uncertainty accumulates sub-linearly
(standard result for volatility processes). A strong R² supports the √T parametrisation;
a weak R² suggests bucket-specific calibration is necessary.

In [ ]:
T_VALS     = np.array(T_GRID, dtype=float)
cstar_vals = np.array(
    [calib_df.loc[lbl, 'c_star'] if lbl in calib_df.index else np.nan
     for lbl in T_LABELS], dtype=float,
)
cstar_by_mat = pd.DataFrame({
    'maturity': T_LABELS,
    'T':        T_VALS,
    'sqrt_T':   np.sqrt(T_VALS),
    'c_star':   cstar_vals,
    'n_val':    [int(calib_df.loc[lbl, 'n_val']) if lbl in calib_df.index else 0
                 for lbl in T_LABELS],
})
cstar_by_mat.to_csv(OUTPUT_DIR / f'{NB}_cstar_term_structure.csv', index=False)

valid_mask = np.isfinite(cstar_vals)
a_hat = b_hat = r2 = np.nan
sqrtT_fit = cstar_fit = np.array([])
if valid_mask.sum() >= 3:
    sqrtT_v = np.sqrt(T_VALS[valid_mask])
    cstar_v = cstar_vals[valid_mask]
    X_reg   = np.column_stack([np.ones(len(sqrtT_v)), sqrtT_v])
    coeffs, _, _, _ = np.linalg.lstsq(X_reg, cstar_v, rcond=None)
    a_hat, b_hat    = coeffs
    sqrtT_fit = np.linspace(0, np.sqrt(T_VALS.max()) * 1.1, 100)
    cstar_fit = a_hat + b_hat * sqrtT_fit
    ss_res = np.sum((cstar_v - (a_hat + b_hat * sqrtT_v)) ** 2)
    ss_tot = np.sum((cstar_v - cstar_v.mean()) ** 2)
    r2     = 1.0 - ss_res / ss_tot if ss_tot > 1e-15 else np.nan
    print(f'OLS: c*(T) = {a_hat:.3f} + {b_hat:.3f} × √T   (R² = {r2:.3f})')

plot_cstar_term_structure(cstar_by_mat, a_hat, b_hat, r2,
                          sqrtT_fit, cstar_fit, PLOT_DIR, prefix=NB)
print(cstar_by_mat.round(3).to_string(index=False))

if valid_mask.sum() >= 3:
    diffs = np.diff(cstar_vals[valid_mask])
    if np.all(diffs > 0):
        print('\nc* strictly increasing in T — uncertainty accumulates with horizon.')
    elif np.all(diffs < 0):
        print('\nc* strictly decreasing in T — short-dated RV forecasts understate tail risk.')
    else:
        _pk = T_LABELS[np.where(valid_mask)[0][int(np.argmax(cstar_vals[valid_mask]))]]
        print(f'\nc* non-monotone (peak at {_pk}) — bucket-specific calibration warranted.')
    if np.isfinite(r2):
        tag = 'strong' if r2 > 0.80 else ('moderate' if r2 > 0.40 else 'weak')
        print(f'  √T regression fit: {tag} (R²={r2:.2f})')


## 6. Coverage Evaluation

```
spread_final(k,T,t) = spread_AS(k,T,t) + c*(bucket) × RV̂(k,T,t)
```

**Coverage** = P(future_rv_5d ≤ spread) on the **test set** (held out from all calibration).
All spreads are in **IV units**; this is not a dollar P&L backtest.

| Column | Definition |
|--------|-----------|
| `coverage_AS` | P(future_rv_5d ≤ spread_AS) — vol-scaled baseline alone |
| `coverage` | P(future_rv_5d ≤ spread_final) — baseline + HAR-J addon |
| `coverage_improvement` | coverage − coverage_AS |

`coverage_improvement > 0` quantifies the incremental residual risk coverage from the
HAR-J addon. Its magnitude depends on how tight the baseline is. **Section 7 shows this
dependence systematically** — it is the main structural result of the framework.

In [ ]:
spread_final, addon_panel = compute_spread_final(
    iv_panel, GRID_COLS, MAT_T_MAP, spread_AS, fc_test, calib_df, log_ewma
)
spread_final.to_csv(OUTPUT_DIR / f'{NB}_spread_final.csv')

bt_df = backtest_spread_coverage(
    split_test, calib_targets, targets_df, spread_final, spread_AS,
    grid_cols=GRID_COLS, mat_t_map=MAT_T_MAP,
)
bt_df.to_csv(OUTPUT_DIR / f'{NB}_mm_backtest.csv', index=False)

print('=== COVERAGE EVALUATION (test set — IV units) ===')
_bt_cols = ['bucket', 'n', 'coverage_AS', 'coverage', 'coverage_improvement',
            'exceedance', 'mean_AS', 'mean_addon', 'addon_share']
print(bt_df[[c for c in _bt_cols if c in bt_df.columns]].round(4).to_string(index=False))

plot_spread_decomposition(split_test, targets_df, spread_final, spread_AS, bt_df,
                          PLOT_DIR, h_main=H_MAIN, prefix=NB)


In [ ]:
ORDER = T_LABELS + ['global']
try:
    regime_test = regime_s.reindex(split_test)
    _rh = float((regime_test == 2).mean()) if len(regime_test) > 0 else np.nan
    _rm = float((regime_test == 1).mean()) if len(regime_test) > 0 else np.nan
    _rl = float((regime_test == 0).mean()) if len(regime_test) > 0 else np.nan
except NameError:
    regime_test = pd.Series(dtype=float)
    _rh = _rm = _rl = np.nan

result_rows = []
for bucket_key in ORDER:
    row_bt = bt_df[bt_df['bucket'] == bucket_key]
    if row_bt.empty:
        continue
    rb       = row_bt.iloc[0].to_dict()
    c_star_v = (float(calib_df.loc[bucket_key, 'c_star'])
                if bucket_key in calib_df.index else np.nan)
    result_rows.append({
        'bucket':               bucket_key,
        'n':                    int(rb['n']),
        'coverage_AS':          rb.get('coverage_AS', np.nan),
        'coverage':             rb['coverage'],
        'coverage_improvement': rb.get('coverage_improvement', np.nan),
        'exceedance':           rb['exceedance'],
        'mean_spread_AS':       rb['mean_AS'],
        'mean_addon':           rb['mean_addon'],
        'mean_spread_final':    rb['mean_final'],
        'addon_share':          rb.get('addon_share', np.nan),
        'c_star':               c_star_v,
        'regime_high_pct':      _rh * 100 if np.isfinite(_rh) else np.nan,
        'regime_mid_pct':       _rm * 100 if np.isfinite(_rm) else np.nan,
        'regime_low_pct':       _rl * 100 if np.isfinite(_rl) else np.nan,
    })

results_df = pd.DataFrame(result_rows)
results_df.to_csv(OUTPUT_DIR / f'{NB}_main_results_table.csv', index=False)

print('=== MAIN RESULTS TABLE (test set) ===')
_cols = ['bucket', 'n', 'coverage_AS', 'coverage', 'coverage_improvement',
         'mean_spread_AS', 'mean_addon', 'addon_share', 'c_star']
print(results_df[[c for c in _cols if c in results_df.columns]].round(4).to_string(index=False))

# ── Stacked bar chart ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Spread Decomposition by Maturity Bucket (test set)', fontsize=11, fontweight='bold')
x        = np.arange(len(results_df))
lbls     = results_df['bucket'].values
as_v     = results_df['mean_spread_AS'].values
add_v    = results_df['mean_addon'].values
cov_as_v = results_df['coverage_AS'].values
cov_v    = results_df['coverage'].values
rrs_v    = results_df['addon_share'].values

ax = axes[0]
ax.bar(x, as_v,  0.55, label='Baseline AS',  color='steelblue', alpha=0.85)
ax.bar(x, add_v, 0.55, bottom=as_v,          label='Addon c×RV̂', color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(lbls, fontsize=9)
ax.set_ylabel('vol-units'); ax.set_title('Stacked Spread Decomposition')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
for xi, (a, b) in enumerate(zip(as_v, add_v)):
    total = a + b; share = b / (total + 1e-12) * 100
    ax.text(xi, total + max(total * 0.02, 1e-5),
            f'{share:.0f}%', ha='center', va='bottom', fontsize=8,
            color='darkorange', fontweight='bold')

ax = axes[1]
w = 0.22
ax.bar(x - w, cov_as_v, w, label='Coverage (AS baseline)', color='lightsteelblue', alpha=0.85)
ax.bar(x,     cov_v,    w, label='Coverage (spread_final)', color='seagreen', alpha=0.85)
ax.bar(x + w, rrs_v,    w, label='Addon share', color='darkorange', alpha=0.85)
ax.axhline(0.95, ls='--', color='red', lw=1.1, label='95% target')
ax.set_xticks(x); ax.set_xticklabels(lbls, fontsize=9)
ax.set_title('Coverage: Baseline vs Final & Addon Share')
ax.legend(fontsize=8); ax.set_ylim(0, 1.1); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / f'{NB}_spread_term_structure.png', dpi=150, bbox_inches='tight')
plt.show()

high_addon = results_df[results_df['addon_share'] > 0.5]
if len(high_addon) > 0:
    print(f'\n{len(high_addon)} bucket(s) where addon > 50% of total spread:')
    for _, r in high_addon.iterrows():
        print(f'  {r["bucket"]:6s}: addon_share={r["addon_share"]*100:.0f}%  c*={r["c_star"]:.2f}')
    print('  HAR-J addon > 50% of spread_final in these buckets (at γ=0.5 baseline).')
    print('  This reflects the baseline being tight relative to realised surface risk.')
else:
    print('\nBaseline dominates in all buckets (addon share ≤ 50%) at γ=0.5.')


The results above are for the fixed baseline `GAMMA_AS = 0.5`. Both `coverage_improvement`
and `addon_share` depend structurally on the baseline's width.

**Section 7 quantifies this dependence across γ ∈ {0.25, 0.5, 1.0, 2.0}.**
That analysis is the main structural result of the framework.

## 7. Baseline Sensitivity (γ) — Main Result

This section addresses the central identification question:

> *How much does the addon's contribution depend on the aggressiveness of the baseline?*

For each γ ∈ {0.25, 0.5, 1.0, 2.0}, the full calibration-evaluation pipeline is re-run:
1. `spread_AS` recomputed with the new γ (same `KAPPA_K`, `DT`).
2. `c*` recalibrated on the **validation set** using the same ES₉₅ procedure.
3. `spread_final` evaluated on the **test set**.

`GAMMA_AS = 0.5` (Section 6) is the baseline case. The other values trace the structural
relationship between γ, `c*`, and `coverage_improvement`.

**Economic mechanism:**
As γ increases, `spread_AS` widens and the residual `max(actual_rv − spread_AS, 0)` shrinks.
The ES₉₅ objective then calibrates a smaller `c*`. At sufficiently large γ, the baseline
alone satisfies the 95% coverage target and the addon becomes economically irrelevant.

The **addon is a residual surface-risk corrector**: its value is maximal under tight
competitive quoting (small γ) and collapses as the baseline grows conservative.
This sharp non-linear transition is the main empirical finding of the framework.

In [ ]:
GAMMA_GRID = [0.25, 0.5, 1.0, 2.0]
gamma_rows = []

print(f'{"γ":>5}  {"c*(global)":>10}  {"cov_AS(global)":>15}  {"cov(global)":>12}  {"improve":>8}')
print('-' * 58)
for gamma_val in GAMMA_GRID:
    sp_as_g = compute_avellaneda_style_spread(
        iv_panel, GRID_COLS, k_flat, MAT_T_MAP, gamma_val, KAPPA_K, DT
    )
    calib_targets_g = {'global': ('future_rv_5d', sp_as_g.mean(axis=1))}
    for t_lbl, t_str in MAT_T_MAP.items():
        tgt_col  = f'future_rv_{H_MAIN}d_{t_lbl}'
        mat_cols = [c for c in GRID_COLS if f'_T_{t_str}' in c]
        if tgt_col in targets_df.columns and mat_cols:
            calib_targets_g[t_lbl] = (tgt_col, sp_as_g[mat_cols].mean(axis=1))

    calib_g = calibrate_c_es95(
        split_val, targets_df, calib_targets_g, fc_val, log_ewma, CALIB_ALPHA
    )
    sf_g, _ = compute_spread_final(
        iv_panel, GRID_COLS, MAT_T_MAP, sp_as_g, fc_test, calib_g, log_ewma
    )
    bt_g = backtest_spread_coverage(
        split_test, calib_targets_g, targets_df, sf_g, sp_as_g,
        grid_cols=GRID_COLS, mat_t_map=MAT_T_MAP,
    )
    for _, row in bt_g.iterrows():
        gamma_rows.append({
            'gamma_AS':             gamma_val,
            'bucket':               row['bucket'],
            'coverage_AS':          row.get('coverage_AS', np.nan),
            'coverage':             row['coverage'],
            'coverage_improvement': row.get('coverage_improvement', np.nan),
            'mean_AS':              float(row['mean_AS']),
            'mean_addon':           float(row['mean_addon']),
            'addon_share':          row.get('addon_share', np.nan),
        })
    _g_row = bt_g[bt_g['bucket'] == 'global'].iloc[0]
    print(f'{gamma_val:>5.2f}  {calib_g.loc["global","c_star"]:>10.3f}  '
          f'{_g_row.get("coverage_AS", np.nan):>15.3f}  '
          f'{_g_row["coverage"]:>12.3f}  '
          f'{_g_row.get("coverage_improvement", np.nan):>8.3f}')

gamma_df = pd.DataFrame(gamma_rows)
gamma_df.to_csv(OUTPUT_DIR / f'{NB}_gamma_sensitivity.csv', index=False)
print(f'\nSaved: {NB}_gamma_sensitivity.csv  ({len(gamma_df)} rows)')


In [ ]:
BUCKET_ORDER  = T_LABELS + ['global']
BUCKET_COLORS = dict(zip(
    BUCKET_ORDER,
    ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b'],
))
GAMMA_MARKERS = {0.25: 'o', 0.5: 's', 1.0: '^', 2.0: 'D'}

# ── Coverage improvement and baseline vs final ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('γ Sensitivity — Coverage', fontsize=11, fontweight='bold')

ax = axes[0]
for bucket in BUCKET_ORDER:
    sub = gamma_df[gamma_df['bucket'] == bucket]
    ax.plot(sub['gamma_AS'], sub['coverage_improvement'],
            marker='o', label=bucket, color=BUCKET_COLORS.get(bucket))
ax.axhline(0, ls='--', color='grey', lw=0.8)
ax.set_xlabel('γ  (GAMMA_AS)'); ax.set_ylabel('coverage_improvement')
ax.set_title('Coverage Improvement  (coverage_final − coverage_AS)')
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
ax.set_xscale('log'); ax.set_xticks([0.25, 0.5, 1.0, 2.0])
ax.set_xticklabels(['0.25', '0.5', '1.0', '2.0'])

ax = axes[1]
x = np.arange(len(BUCKET_ORDER))
for gamma_val in GAMMA_GRID:
    sub = (gamma_df[gamma_df['gamma_AS'] == gamma_val]
           .set_index('bucket').reindex(BUCKET_ORDER))
    mk  = GAMMA_MARKERS[gamma_val]
    ax.plot(x, sub['coverage_AS'].values, ls='--', alpha=0.65, marker=mk,
            color=f'C{GAMMA_GRID.index(gamma_val)}',
            label=f'γ={gamma_val} AS')
    ax.plot(x, sub['coverage'].values,   ls='-',  alpha=0.90, marker=mk,
            color=f'C{GAMMA_GRID.index(gamma_val)}',
            label=f'γ={gamma_val} final')
ax.axhline(0.95, ls=':', color='red', lw=1.0, label='95%')
ax.set_xticks(x); ax.set_xticklabels(BUCKET_ORDER, fontsize=8)
ax.set_ylabel('coverage'); ax.set_title('coverage_AS vs coverage_final  (all γ)')
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(PLOT_DIR / f'{NB}_gamma_sensitivity_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Addon share vs gamma per maturity ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle('γ Sensitivity — Addon Share', fontsize=11, fontweight='bold')
for bucket in BUCKET_ORDER:
    sub = gamma_df[gamma_df['bucket'] == bucket]
    ax.plot(sub['gamma_AS'], sub['addon_share'],
            marker='o', label=bucket, color=BUCKET_COLORS.get(bucket))
ax.axhline(0.5, ls='--', color='grey', lw=0.8, label='50%')
ax.set_xlabel('γ  (GAMMA_AS)')
ax.set_ylabel('addon_share = mean_addon / mean_final')
ax.set_title('Addon Share vs γ  (higher γ → wider baseline → addon share ↓)')
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
ax.set_xscale('log'); ax.set_xticks([0.25, 0.5, 1.0, 2.0])
ax.set_xticklabels(['0.25', '0.5', '1.0', '2.0']); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(PLOT_DIR / f'{NB}_gamma_sensitivity_addon_share.png', dpi=150, bbox_inches='tight')
plt.show()


### Interpretation

The sensitivity table makes the structural relationship explicit:

- **γ = 0.25 (very tight):** `coverage_AS ≈ 16%` — the baseline leaves most surface risk
  unhedged. The addon compensates almost entirely; `c* ≈ 4`. The coverage improvement
  is large but the baseline is unrealistically narrow for any practical quoting rule.

- **γ = 0.50 (reference case):** `coverage_AS ≈ 61%` — moderate baseline coverage.
  The addon raises coverage to ≈96%; `c* ≈ 3.4`. Coverage improvement ≈ 35pp.
  This is the case analysed in Section 6.

- **γ = 1.00 (wider baseline):** `coverage_AS ≈ 87%` — baseline is reasonably protective.
  The addon adds ≈8pp; `c*` collapses to ≈0.1.

- **γ = 2.00 (conservative):** `coverage_AS ≈ 96%` — the baseline alone satisfies the
  95% target. `c* ≈ 0`; the addon is economically irrelevant.

**The addon is a residual surface-risk corrector.** Its contribution to coverage scales
inversely with γ: the addon adds value precisely when the market maker quotes tightly.
This is an economically coherent result, not a weakness of the framework.

**Short-maturity pattern:** If short-dated buckets (1M, 3M) show consistently higher
`addon_share` across all γ, the pattern is structural — short-term surface risk is harder
to absorb with a vol-scaled ATM baseline, leaving more residual for the HAR-J forecast.

The sharp non-linear transition between γ = 0.5 and γ = 1.0 — where `c*` collapses from
≈3.4 to ≈0.1 — is the key empirical finding. It establishes the regime boundary at which
IV-surface RV forecasting ceases to add material coverage.

## 8. Summary and Conclusions

In [ ]:
SEP = '=' * 70
print(SEP)
print('D — SSVI IV-SURFACE RISK FRAMEWORK — SUMMARY')
print(SEP)
print(f'''
FRAMEWORK
---------
Risk-adjusted quoting overlay based on IV-surface RV forecasting.
NOT a full market-making model: no inventory, order flow, or bid-ask calibration.

SAMPLE
------
  IV panel   : {iv_panel.index[0].date()} → {iv_panel.index[-1].date()}
               ({len(iv_panel)} trading days, {len(GRID_COLS)} grid points)
  Exclusions : {len(_raw) - (_raw["n_obs"] > 0).sum()} dates with n_obs < 100 (Section 2)
  Train      : {split_train[0].date()} → {split_train[-1].date()} ({len(split_train)} days, 70%)
  Validation : {split_val[0].date()}  → {split_val[-1].date()}  ({len(split_val)} days, 15%)
  Test       : {split_test[0].date()}  → {split_test[-1].date()}  ({len(split_test)} days, 15%)
''')

print('JUMP DETECTION')
print(f'  Selected: Method {selected_key}  Rate: {selected_pct:.1f}%  (BPV baseline: {old_jump_pct:.1f}%)')

print('\nFORECAST TABLE (global surface RV, h=5, test set):')
_cols = ['model', 'n', 'MAE_log', 'RMSE_log', 'QLIKE', 'DM_stat_vs_HAR', 'DM_pval_vs_HAR']
print(metrics_df[metrics_df['target'] == 'global']
      [[c for c in _cols if c in metrics_df.columns]].round(4).to_string(index=False))

print('\nBASELINE SPREAD PARAMETERS (heuristic, not calibrated to market data):')
print(f'  GAMMA_AS={GAMMA_AS}  KAPPA_K={KAPPA_K}  DT=1/252')

print('\nc* CALIBRATION (val set, ES₉₅ of residual risk ratio):')
print(calib_df.round(3).to_string())

print('\nCOVERAGE EVALUATION (test set — spread in IV units, not dollar P&L):')
_bt_cols = ['bucket', 'coverage_AS', 'coverage', 'coverage_improvement',
            'exceedance', 'mean_AS', 'mean_addon', 'addon_share', 'n']
print(bt_df[[c for c in _bt_cols if c in bt_df.columns]].round(4).to_string(index=False))

try:
    print('\nVEGA-WEIGHTED ROBUSTNESS (heuristic proxy, not full P&L):')
    print(vw_df[['maturity', 'coverage_unweighted', 'vega_weighted_shortfall', 'n']]
          .round(4).to_string(index=False))
except NameError:
    print('\nVEGA-WEIGHTED ROBUSTNESS: run Appendix A.3 to populate vw_df.')

if 'gamma_df' in vars():
    print('\nBASELINE SENSITIVITY — main structural result (global bucket):')
    _gcols = ['gamma_AS', 'coverage_AS', 'coverage', 'coverage_improvement', 'addon_share']
    print(gamma_df[gamma_df['bucket'] == 'global'][_gcols].round(3).to_string(index=False))
    print('  → Addon is most relevant under tight (small γ) competitive quoting.')
    print('  → At γ ≥ 1.0, the baseline alone provides ≥87% coverage.')
    print('  → c* converges to zero as γ grows: addon is a residual corrector.')

print(f'''
KEY FINDINGS
------------
1. surface_move (IV-surface RV) is persistent and forecastable with HAR-J.
   Jump-component features provide an interpretable structural decomposition
   even when the DM test does not reject equal MSE vs plain HAR.
2. The ES₉₅ calibration produces a c* that decreases monotonically with γ:
   the addon compensates for the gap between the baseline and the tail of
   realised surface risk — and that gap shrinks as the baseline widens.
3. Coverage improvement is maximal under tight competitive quoting (small γ).
   IV-surface RV forecasting adds the most value precisely when the baseline
   is narrow relative to realised surface risk.
4. At γ ≥ 1.0, the addon adds ≤8pp. At γ = 2.0, the baseline alone achieves
   the 95% target and the addon becomes economically irrelevant.

LIMITATIONS
-----------
1. No inventory dynamics — net vega/gamma position not modelled.
2. No order-flow modelling — adverse selection and fill rates not considered.
3. No observed bid-ask calibration — γ and κ are heuristic, not estimated.
4. No dollar P&L — coverage is in IV units, not revenue.
5. Equal-weighted surface_move — illiquid OTM points weighted equally to ATM.
6. c* estimated from a single validation window; tail sample ≈ 19 obs for ES₉₅.
7. The framework should be interpreted as a risk-adjusted quoting overlay,
   not a complete market-making model.
''')
print(SEP); print('Done.'); print(SEP)


---

## Appendix — Supplementary Descriptive Analysis

> The following sections are **not** part of the core pipeline.
> PCA, HMM, and vega-weighted robustness do not feed into the spread formula,
> the c* calibration, or the coverage evaluation.
> They are included to characterise the IV-surface factor structure, vol-of-vol
> regime dynamics, and approximate economic weighting of coverage shortfalls.
> Results in Sections 1–8 are fully independent of whether these cells are run.

## Appendix A.1 — PCA on ΔIV

Standard-scaler and PCA fitted on the **train set only**; projected onto val and test.
The first three PCs typically correspond to parallel shifts (PC1), skew changes (PC2),
and curvature changes (PC3) of the IV surface (Cont & da Fonseca 2002).

PC scores are saved to `output/cache/` for reference; they are **not used** in the
spread engine or HAR features.

In [ ]:
scaler_pca, pca = fit_train_pca(delta_iv, split_train, n_components=10)
pc_df, loadings_df, ev = transform_pca_splits(
    delta_iv, scaler_pca, pca, split_train, split_val, split_test
)
save_pca_outputs(pc_df, loadings_df, CACHE_DIR, prefix=NB)

cum_ev = ev.cumsum()
print(f'PCA (train-only fit) — {pca.n_components_} components')
print(f'PC1-3 cumulative variance: {cum_ev[min(2, len(ev)-1)]*100:.1f}%')
for i in range(min(5, len(ev))):
    print(f'  PC{i+1}: {ev[i]*100:.1f}%  (cum: {cum_ev[i]*100:.1f}%)')

plot_pca_loadings(loadings_df, K_GRID, T_GRID, T_LABELS, ev, PLOT_DIR, prefix=NB)
plot_pca_scores(pc_df, PLOT_DIR, prefix=NB)


## Appendix A.2 — HMM Regime Detection

3-state Gaussian HMM on `log(RV₂₂d)`, fitted on **train+val** only.
Regimes sorted by mean log-RV: 0 = low-vol, 1 = mid-vol, 2 = high-vol.
Regime assignments saved to `output/cache/`.

⚠ The Viterbi path is **retrospective**: it uses the full series for state inference.
Regime labels are not available in real time and cannot be used as trading signals.
They are **not used** in the spread formula or coverage evaluation.

In [ ]:
log_rv22_full = np.log(np.sqrt(sm_sq.rolling(22, min_periods=10).mean()) + 1e-8)
hmm, regime_s, remap, trans_df = fit_hmm_regimes(log_rv22_full, split_train, split_val)

REGIME_LABELS = {0: 'Low-vol', 1: 'Mid-vol', 2: 'High-vol'}
summary_hmm   = summarize_hmm_regimes(regime_s, log_rv22_full, REGIME_LABELS)
regime_s.to_frame().to_csv(CACHE_DIR / f'{NB}_hmm_regimes.csv')

print('HMM Regime Summary:')
print(summary_hmm.round(3).to_string(index=False))
print('\nTransition matrix (rows=from, cols=to):')
print(trans_df.round(3).to_string())

plot_hmm_regimes(smove, regime_s, REGIME_LABELS, PLOT_DIR, prefix=NB)


## Appendix A.3 — Vega-Weighted Robustness Check

Spreads and RV targets are in vol-units. Economic P&L impact scales with option vega:
a shortfall on a high-vega contract is more costly than on a low-vega one.
This section applies a heuristic vega approximation (`v ≈ φ(d₁) × √T`) to check whether
shortfall days are concentrated in economically sensitive surface regions.

This is an approximate robustness check, not a full portfolio P&L analysis.
A complete assessment would require actual portfolio positions and delta-hedging P&L.

In [ ]:
vw_df = compute_vega_weighted_robustness(
    iv_panel, spread_final, targets_df, GRID_COLS, MAT_T_MAP, split_test, H_MAIN
)
vw_df.to_csv(OUTPUT_DIR / f'{NB}_vega_weighted_robustness.csv', index=False)

print('=== VEGA-WEIGHTED ROBUSTNESS (test set) ===')
print(vw_df.round(4).to_string(index=False))


In [ ]:
# ── Final validation ──────────────────────────────────────────────────────────
print('=== FINAL VALIDATION ===')
print(f'params_ok : {len(params_ok)} rows  '
      f'DatetimeIndex={isinstance(params_ok.index, pd.DatetimeIndex)}  '
      f'monotone={params_ok.index.is_monotonic_increasing}')
print(f'iv_panel  : {iv_panel.shape}  '
      f'index_match_params={iv_panel.index.equals(params_ok.index)}')
print(f'delta_iv  : {delta_iv.shape}  '
      f'index_match_iv1={delta_iv.index.equals(iv_panel.index[1:])}')
print(f'Train : {len(split_train)} days  [{split_train[0].date()} → {split_train[-1].date()}]')
print(f'Val   : {len(split_val)} days  [{split_val[0].date()} → {split_val[-1].date()}]')
print(f'Test  : {len(split_test)} days  [{split_test[0].date()} → {split_test[-1].date()}]')
assert split_train[-1] < split_val[0] and split_val[-1] < split_test[0], \
    'Chronological split violated — leakage detected'
print('Chronological split: OK (no leakage)')
print()
print('Output files (OUTPUT_DIR):')
for _f in sorted(OUTPUT_DIR.glob(f'{NB}_*.csv')):
    print(f'  {_f.name}')
print()
print('Cache files (CACHE_DIR):')
for _f in sorted(CACHE_DIR.glob('*.csv')):
    print(f'  {_f.name}')
